# Combinación de DataFrames: Merge, Concat y Join

**Curso:** Python para Ciencia de Datos \
**Fecha:** 16 de octubre de 2025 \
**Ruta:** Fundamentos de Data Science e IA \
**Repositorio:** `bootcamp-fundamentos-ciencia-de-datos`

## 📋 Índice

1. [Concepto de Combinación de DataFrames](#concepto)
2. [Merge - Unión por Columnas Clave](#merge)
3. [Concat - Concatenación Simple](#concat)
4. [Join - Unión por Índice](#join)
5. [Casos de Uso en Data Science](#casos-uso)
6. [Recursos Adicionales](#recursos)

## Concepto de Combinación de DataFrames

En análisis de datos real, la información suele estar **distribuida en múltiples tablas**. Combinarlas correctamente es fundamental.

### ¿Cuándo necesitas combinar DataFrames?

- 📊 Datos de clientes en una tabla, transacciones en otra
- 🏢 Información de productos separada de inventario
- 📅 Datos históricos divididos por períodos
- 🌍 Información geográfica complementaria

### Tres métodos principales

| Método | Cuándo usar | Clave de unión |
|--------|-------------|----------------|
| **merge()** | Unir por columnas comunes | Columnas con valores coincidentes |
| **concat()** | Apilar DataFrames | Sin clave (concatenación simple) |
| **join()** | Unir por índices | Índices de los DataFrames |

### Analogía SQL
```
merge()  = SQL JOIN
concat() = SQL UNION
join()   = merge() usando índices
```

In [18]:
import pandas as pd
import numpy as np

---

## Merge - Unión por Columnas Clave

`merge()` es equivalente a **JOIN en SQL**. Une DataFrames basándose en valores coincidentes en columnas específicas.

### Tipos de Merge (como JOINs en SQL)
```
INNER JOIN          LEFT JOIN           RIGHT JOIN         OUTER JOIN
┌───┬───┐           ┌───┬───┐           ┌───┬───┐          ┌───┬───┐
│ A │ B │           │ A │ B │           │ A │ B │          │ A │ B │
├───┼───┤           ├───┼───┤           ├───┼───┤          ├───┼───┤
│ B │ C │    →      │ B │ C │    →      │ B │ C │    →     │ B │ C │
└───┴───┘           │ C │   │           │   │ D │          │ C │ D │
                    └───┴───┘           └───┴───┘          └───┴───┘
Solo coincidencias  Todo left + matches  Todo right + matches  Todo de ambos
```

### Sintaxis básica
```python
pd.merge(left_df, right_df, on='columna_clave', how='tipo_join')
```

In [19]:
# Crear DataFrames de ejemplo
df1 = pd.DataFrame({
    'key': ['A', 'B', 'C'],
    'value_1': [1, 2, 3]
})

df2 = pd.DataFrame({
    'key': ['B', 'C', 'D'],
    'value_2': [4, 5, 6]
})

print("📊 DataFrame 1:")
print(df1)
print("\n📊 DataFrame 2:")
print(df2)

📊 DataFrame 1:
  key  value_1
0   A        1
1   B        2
2   C        3

📊 DataFrame 2:
  key  value_2
0   B        4
1   C        5
2   D        6


### Inner Join (Intersección)

Retorna **solo las filas con coincidencias en ambos DataFrames**.

In [20]:
# Inner merge - Solo coincidencias
inner_merged = pd.merge(df1, df2, on='key', how='inner')

print("🔗 Inner Merge (solo B y C tienen coincidencia):")
print(inner_merged)

🔗 Inner Merge (solo B y C tienen coincidencia):
  key  value_1  value_2
0   B        2        4
1   C        3        5


### Outer Join (Unión)

Retorna **todas las filas de ambos DataFrames**, llenando con NaN donde no hay coincidencia.

In [21]:
# Outer merge - Todo de ambos
outer_merged = pd.merge(df1, df2, on='key', how='outer')

print("🔗 Outer Merge (A, B, C, D - todos):")
print(outer_merged)

🔗 Outer Merge (A, B, C, D - todos):
  key  value_1  value_2
0   A      1.0      NaN
1   B      2.0      4.0
2   C      3.0      5.0
3   D      NaN      6.0


### Left Join

Retorna **todas las filas del DataFrame izquierdo** + coincidencias del derecho.

In [22]:
# Left merge - Todo de df1 + matches de df2
left_merged = pd.merge(df1, df2, on='key', how='left')

print("🔗 Left Merge (todas las filas de df1):")
print(left_merged)

🔗 Left Merge (todas las filas de df1):
  key  value_1  value_2
0   A        1      NaN
1   B        2      4.0
2   C        3      5.0


### Right Join

Retorna **todas las filas del DataFrame derecho** + coincidencias del izquierdo.

In [23]:
# Right merge - Todo de df2 + matches de df1
right_merged = pd.merge(df1, df2, on='key', how='right')

print("🔗 Right Merge (todas las filas de df2):")
print(right_merged)

🔗 Right Merge (todas las filas de df2):
  key  value_1  value_2
0   B      2.0        4
1   C      3.0        5
2   D      NaN        6


### Comparación visual de los 4 tipos

| Tipo | Filas resultantes | Cuándo usar |
|------|------------------|-------------|
| **inner** | Solo coincidencias | Análisis donde ambos datos deben existir |
| **outer** | Todo de ambos | No quieres perder ningún dato |
| **left** | Todo del primero | El primer DataFrame es la "base" |
| **right** | Todo del segundo | El segundo DataFrame es la "base" |

### Sufijos para columnas duplicadas

Cuando ambos DataFrames tienen columnas con el mismo nombre (no son la clave):

In [24]:
# Crear DataFrames con columnas del mismo nombre
df_a = pd.DataFrame({
    'key': ['A', 'B', 'C'],
    'value': [1, 2, 3]  # Mismo nombre
})

df_b = pd.DataFrame({
    'key': ['B', 'C', 'D'],
    'value': [4, 5, 6]  # Mismo nombre
})

# Merge con sufijos personalizados
merged_custom = pd.merge(
    df_a, df_b,
    on='key',
    how='outer',
    suffixes=('_left', '_right')
)

print("🔗 Merge con sufijos personalizados:")
print(merged_custom)

🔗 Merge con sufijos personalizados:
  key  value_left  value_right
0   A         1.0          NaN
1   B         2.0          4.0
2   C         3.0          5.0
3   D         NaN          6.0


---

## Concat - Concatenación Simple

`concat()` **apila** DataFrames vertical u horizontalmente **sin buscar coincidencias**.

### Concatenación vertical (por defecto)
```
df1        df2           concat([df1, df2])
┌───┬───┐  ┌───┬───┐     ┌───┬───┐
│ A │ B │  │ A │ B │     │ A │ B │
├───┼───┤  ├───┼───┤  →  ├───┼───┤
│ 1 │ 2 │  │ 4 │ 5 │     │ 1 │ 2 │
│ 2 │ 3 │  │ 5 │ 6 │     │ 2 │ 3 │
└───┴───┘  └───┴───┘     │ 4 │ 5 │
                         │ 5 │ 6 │
                         └───┴───┘
```

### Sintaxis
```python
pd.concat([df1, df2], axis=0)  # Vertical (por defecto)
pd.concat([df1, df2], axis=1)  # Horizontal
```

In [25]:
# Crear DataFrames para concatenar
df3 = pd.DataFrame({
    'A': ['A0', 'A1', 'A2'],
    'B': ['B0', 'B1', 'B2']
})

df4 = pd.DataFrame({
    'A': ['A3', 'A4', 'A5'],
    'B': ['B3', 'B4', 'B5']
})

print("📊 DataFrame 3:")
print(df3)
print("\n📊 DataFrame 4:")
print(df4)

📊 DataFrame 3:
    A   B
0  A0  B0
1  A1  B1
2  A2  B2

📊 DataFrame 4:
    A   B
0  A3  B3
1  A4  B4
2  A5  B5


### Concatenación vertical (axis=0)

Apila un DataFrame **debajo** del otro.

In [26]:
# Concatenar verticalmente (apilar filas)
vertical_concat = pd.concat([df3, df4])

print("⬇️ Concatenación vertical:")
print(vertical_concat)
print(f"\nNota: Índices se repiten (0, 1, 2, 0, 1, 2)")

⬇️ Concatenación vertical:
    A   B
0  A0  B0
1  A1  B1
2  A2  B2
0  A3  B3
1  A4  B4
2  A5  B5

Nota: Índices se repiten (0, 1, 2, 0, 1, 2)


In [27]:
# Resetear índices después de concat
vertical_concat_reset = pd.concat([df3, df4], ignore_index=True)

print("⬇️ Concatenación vertical con índices reseteados:")
print(vertical_concat_reset)

⬇️ Concatenación vertical con índices reseteados:
    A   B
0  A0  B0
1  A1  B1
2  A2  B2
3  A3  B3
4  A4  B4
5  A5  B5


### Concatenación horizontal (axis=1)

Coloca un DataFrame **al lado** del otro.

In [28]:
# Concatenar horizontalmente (lado a lado)
horizontal_concat = pd.concat([df3, df4], axis=1)

print("➡️ Concatenación horizontal:")
print(horizontal_concat)
print("\nNota: Columnas duplicadas (A, B, A, B)")

➡️ Concatenación horizontal:
    A   B   A   B
0  A0  B0  A3  B3
1  A1  B1  A4  B4
2  A2  B2  A5  B5

Nota: Columnas duplicadas (A, B, A, B)


### Diferencias clave: merge vs concat

| Aspecto | merge() | concat() |
|---------|---------|----------|
| **Búsqueda de coincidencias** | ✅ Sí (por columna clave) | ❌ No |
| **Uso típico** | Combinar tablas relacionadas | Apilar datos similares |
| **Equivalente SQL** | JOIN | UNION |
| **Complejidad** | Mayor (busca matches) | Menor (simple apilado) |

---

## Join - Unión por Índice

`join()` es similar a `merge()` pero **usa índices** en lugar de columnas para la unión.

### Sintaxis
```python
df1.join(df2, how='tipo')
```

Es equivalente a:
```python
pd.merge(df1, df2, left_index=True, right_index=True, how='tipo')
```

In [29]:
# Crear DataFrames con índices personalizados
df5 = pd.DataFrame({
    'A': ['A0', 'A1', 'A2'],
    'B': ['B0', 'B1', 'B2']
}, index=['K0', 'K1', 'K2'])

df6 = pd.DataFrame({
    'C': ['C0', 'C1', 'C2'],
    'D': ['D0', 'D1', 'D2']
}, index=['K0', 'K2', 'K3'])

print("📊 DataFrame 5 (índices: K0, K1, K2):")
print(df5)
print("\n📊 DataFrame 6 (índices: K0, K2, K3):")
print(df6)

📊 DataFrame 5 (índices: K0, K1, K2):
     A   B
K0  A0  B0
K1  A1  B1
K2  A2  B2

📊 DataFrame 6 (índices: K0, K2, K3):
     C   D
K0  C0  D0
K2  C1  D1
K3  C2  D2


In [30]:
# Inner join - Solo índices comunes (K0, K2)
joined_inner = df5.join(df6, how='inner')

print("🔗 Inner Join (índices K0 y K2):")
print(joined_inner)

🔗 Inner Join (índices K0 y K2):
     A   B   C   D
K0  A0  B0  C0  D0
K2  A2  B2  C1  D1


In [31]:
# Outer join - Todos los índices
joined_outer = df5.join(df6, how='outer')

print("🔗 Outer Join (todos los índices K0, K1, K2, K3):")
print(joined_outer)

🔗 Outer Join (todos los índices K0, K1, K2, K3):
      A    B    C    D
K0   A0   B0   C0   D0
K1   A1   B1  NaN  NaN
K2   A2   B2   C1   D1
K3  NaN  NaN   C2   D2


In [32]:
# Left join - Todos del df5
joined_left = df5.join(df6, how='left')

print("🔗 Left Join (todos los índices de df5):")
print(joined_left)

🔗 Left Join (todos los índices de df5):
     A   B    C    D
K0  A0  B0   C0   D0
K1  A1  B1  NaN  NaN
K2  A2  B2   C1   D1


### Cuándo usar join() vs merge()

| Situación | Recomendación |
|-----------|---------------|
| **Índices son claves** | `join()` (más simple) |
| **Columnas son claves** | `merge()` |
| **Múltiples columnas clave** | `merge()` |
| **Necesitas control fino** | `merge()` |

---

## Casos de Uso en Data Science

### Caso de Uso 1: Enriquecer Datos de Clientes con Transacciones

**Contexto:** Tienes una tabla de clientes con información demográfica y otra tabla con transacciones. Necesitas combinarlas para análisis de comportamiento.

**Objetivo:** Unir datos de clientes con sus transacciones usando merge.

In [33]:
# Simular datos de clientes
customers = pd.DataFrame({
    'CustomerID': [1001, 1002, 1003, 1004, 1005],
    'Name': ['Ana García', 'Luis Pérez', 'María López', 'Carlos Ruiz', 'Sofía Díaz'],
    'Country': ['Colombia', 'Mexico', 'Colombia', 'Spain', 'Argentina'],
    'Segment': ['Premium', 'Regular', 'Premium', 'VIP', 'Regular']
})

# Simular datos de transacciones
transactions = pd.DataFrame({
    'TransactionID': [101, 102, 103, 104, 105, 106],
    'CustomerID': [1001, 1002, 1001, 1003, 1006, 1002],  # 1006 no existe en customers
    'Amount': [150.00, 75.50, 200.00, 300.00, 50.00, 120.00],
    'Product': ['Laptop', 'Mouse', 'Monitor', 'Keyboard', 'Cable', 'Headphones']
})

print("👥 Tabla de Clientes:")
print(customers)
print("\n💳 Tabla de Transacciones:")
print(transactions)

👥 Tabla de Clientes:
   CustomerID         Name    Country  Segment
0        1001   Ana García   Colombia  Premium
1        1002   Luis Pérez     Mexico  Regular
2        1003  María López   Colombia  Premium
3        1004  Carlos Ruiz      Spain      VIP
4        1005   Sofía Díaz  Argentina  Regular

💳 Tabla de Transacciones:
   TransactionID  CustomerID  Amount     Product
0            101        1001   150.0      Laptop
1            102        1002    75.5       Mouse
2            103        1001   200.0     Monitor
3            104        1003   300.0    Keyboard
4            105        1006    50.0       Cable
5            106        1002   120.0  Headphones


In [34]:
# Inner join - Solo clientes con transacciones
customer_transactions = pd.merge(
    customers,
    transactions,
    on='CustomerID',
    how='inner'
)

print("🔗 Clientes con transacciones (Inner Join):")
print(customer_transactions)
print(f"\nTotal de registros: {len(customer_transactions)}")

🔗 Clientes con transacciones (Inner Join):
   CustomerID         Name   Country  Segment  TransactionID  Amount  \
0        1001   Ana García  Colombia  Premium            101   150.0   
1        1001   Ana García  Colombia  Premium            103   200.0   
2        1002   Luis Pérez    Mexico  Regular            102    75.5   
3        1002   Luis Pérez    Mexico  Regular            106   120.0   
4        1003  María López  Colombia  Premium            104   300.0   

      Product  
0      Laptop  
1     Monitor  
2       Mouse  
3  Headphones  
4    Keyboard  

Total de registros: 5


In [36]:
# Left join - Todos los clientes, con o sin transacciones
all_customers = pd.merge(
    customers,
    transactions,
    on='CustomerID',
    how='left'
)

print("🔗 Todos los clientes + transacciones (Left Join):")
print(all_customers)

# Identificar clientes sin transacciones
no_transactions = all_customers[all_customers['TransactionID'].isna()]
print(f"\n📊 Clientes sin transacciones:")
print(no_transactions[['CustomerID', 'Name', 'Segment']])

🔗 Todos los clientes + transacciones (Left Join):
   CustomerID         Name    Country  Segment  TransactionID  Amount  \
0        1001   Ana García   Colombia  Premium          101.0   150.0   
1        1001   Ana García   Colombia  Premium          103.0   200.0   
2        1002   Luis Pérez     Mexico  Regular          102.0    75.5   
3        1002   Luis Pérez     Mexico  Regular          106.0   120.0   
4        1003  María López   Colombia  Premium          104.0   300.0   
5        1004  Carlos Ruiz      Spain      VIP            NaN     NaN   
6        1005   Sofía Díaz  Argentina  Regular            NaN     NaN   

      Product  
0      Laptop  
1     Monitor  
2       Mouse  
3  Headphones  
4    Keyboard  
5         NaN  
6         NaN  

📊 Clientes sin transacciones:
   CustomerID         Name  Segment
5        1004  Carlos Ruiz      VIP
6        1005   Sofía Díaz  Regular


In [37]:
# Análisis: Gasto total por cliente y segmento
customer_summary = customer_transactions.groupby(['CustomerID', 'Name', 'Segment']).agg({
    'Amount': ['sum', 'count'],
    'TransactionID': 'nunique'
}).round(2)

customer_summary.columns = ['Total_Spent', 'Num_Purchases', 'Unique_Transactions']
customer_summary = customer_summary.reset_index()

print("📊 Resumen por cliente:")
print(customer_summary.sort_values('Total_Spent', ascending=False))

📊 Resumen por cliente:
   CustomerID         Name  Segment  Total_Spent  Num_Purchases  \
0        1001   Ana García  Premium        350.0              2   
2        1003  María López  Premium        300.0              1   
1        1002   Luis Pérez  Regular        195.5              2   

   Unique_Transactions  
0                    2  
2                    1  
1                    2  


**🎯 Insights del Caso 1:**

✅ **Inner join**: 5 transacciones de 3 clientes
✅ **Left join**: Identifica 2 clientes sin transacciones (Carlos y Sofía)
✅ **Cliente 1001 (Ana)**: Mayor gasto con 2 transacciones
✅ **Transacción huérfana**: CustomerID 1006 no existe (se perdió en inner join)

**Decisión de negocio:**
- Contactar a clientes sin transacciones (Carlos y Sofía)
- Investigar la transacción con CustomerID 1006 (error de datos)

### Caso de Uso 2: Consolidar Reportes Mensuales con Concat

**Contexto:** Tienes reportes de ventas mensuales en archivos separados. Necesitas consolidarlos en un único DataFrame para análisis anual.

**Objetivo:** Usar concat para apilar datos de diferentes meses.

In [ ]:
# Simular reportes mensuales
jan_sales = pd.DataFrame({
    'Date': pd.date_range('2024-01-01', periods=5),
    'Product': ['A', 'B', 'A', 'C', 'B'],
    'Revenue': [100, 150, 120, 200, 180]
})

feb_sales = pd.DataFrame({
    'Date': pd.date_range('2024-02-01', periods=5),
    'Product': ['A', 'C', 'B', 'A', 'C'],
    'Revenue': [110, 220, 190, 130, 210]
})

mar_sales = pd.DataFrame({
    'Date': pd.date_range('2024-03-01', periods=5),
    'Product': ['B', 'A', 'C', 'B', 'A'],
    'Revenue': [200, 140, 230, 195, 125]
})

print("📅 Ventas de Enero:")
print(jan_sales)
print("\n📅 Ventas de Febrero:")
print(feb_sales.head())

In [ ]:
# Concatenar todos los meses
q1_sales = pd.concat([jan_sales, feb_sales, mar_sales], ignore_index=True)

print("📊 Ventas Q1 (Enero-Marzo consolidado):")
print(q1_sales)
print(f"\nTotal de registros: {len(q1_sales)}")

In [ ]:
# Añadir columna de mes para análisis
q1_sales['Month'] = q1_sales['Date'].dt.month_name()

# Análisis por mes y producto
monthly_analysis = q1_sales.groupby(['Month', 'Product'])['Revenue'].sum().unstack(fill_value=0)

print("📈 Ingresos por Mes × Producto:")
print(monthly_analysis)

# Totales por mes
print("\n💰 Total por mes:")
print(monthly_analysis.sum(axis=1))

**🎯 Insights del Caso 2:**

✅ **Consolidación exitosa**: 15 registros de 3 meses
✅ **Producto C**: Mayor ingreso en marzo ($230)
✅ **Tendencia**: Ingresos crecientes mes a mes
✅ **Estructura uniforme**: Concat funciona porque todos los meses tienen las mismas columnas

**Aplicación práctica:**
- Automatizar consolidación de reportes mensuales
- Análisis de tendencias temporales
- Preparar datos para dashboards anuales

---

## Recursos Adicionales

### Documentación oficial
- [pandas.merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
- [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [pandas.DataFrame.join](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html)
- [Merge, join, concatenate](https://pandas.pydata.org/docs/user_guide/merging.html)

### Cheat Sheet
```python
# MERGE - Unión por columnas
pd.merge(df1, df2, on='key', how='inner')  # Solo coincidencias
pd.merge(df1, df2, on='key', how='outer')  # Todo de ambos
pd.merge(df1, df2, on='key', how='left')   # Todo de df1
pd.merge(df1, df2, on='key', how='right')  # Todo de df2

# CONCAT - Apilado simple
pd.concat([df1, df2])                      # Vertical
pd.concat([df1, df2], axis=1)              # Horizontal
pd.concat([df1, df2], ignore_index=True)   # Resetear índices

# JOIN - Unión por índices
df1.join(df2, how='inner')                 # Por índice
```

### Tabla de decisión

| Necesitas | Método | Ejemplo |
|-----------|--------|---------|
| Combinar tablas relacionadas | `merge()` | Clientes + Transacciones |
| Apilar datos del mismo tipo | `concat()` | Reportes mensuales |
| Unir por índices | `join()` | Series temporales |
| Múltiples claves | `merge()` con `on=[col1, col2]` | Joins complejos |

